# PASAR DE UN DATASET HORARIO A DIARIO HACIENDO LA MEDIA DIARIA

In [1]:
import pandas as pd

input_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_semicompletos/df__Viver_completo.csv"         # <- pon aquí tu CSV horario
output_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_media_diaria/df__Viver_diario_vfq.csv" # <- nombre del CSV de salida

# leer y parsear datetime
df = pd.read_csv(input_path, parse_dates=["datetime"])

# asegurarse de que datetime sea índice
df = df.set_index("datetime")

# columnas objetivo (se ignorarán si no existen)
cols = ["NO", "NO2", "NOx", "O3"]
available_cols = [c for c in cols if c in df.columns]
if not available_cols:
    raise ValueError("No se encontraron columnas NO, NO2, NOx ni O3 en el CSV de entrada.")

# resample diario y calcular la media (ignora NaNs por defecto)
daily = df[available_cols].resample("D").mean()

# convertir índice a columna 'datetime' con frecuencia diaria (YYYY-MM-DD)
daily = daily.reset_index()
daily["datetime"] = daily["datetime"].dt.strftime("%Y-%m-%d")

# guardar CSV resultante
daily.to_csv(output_path, index=False)
print(f"CSV de medias diarias guardado en: {output_path}")


CSV de medias diarias guardado en: /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_media_diaria/df__Viver_diario_vfq.csv


# CAMBIAR NOMBRE Y UNIDADES DATOS AEMET

In [7]:
import pandas as pd

input_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-AEMET/Datosmeteorologicossegorbe.csv"         # <- pon aquí tu CSV horario
output_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-AEMET/Datosmeteorologicossegorbe_limpios.csv" # <- nombre del CSV de salida


# 1) leer CSV
df = pd.read_csv(input_path)

# 2) convertir 'dir' a numérico (si hay valores no numéricos quedarán NaN)
df['dir'] = pd.to_numeric(df['dir'], errors='coerce')

# 3) pasar dir 0-100 -> 0-360
#    grados = dir * 360 / 100
#    si prefieres que 100 -> 0 (equivalente), usa % 360; aquí dejamos el valor directo
df['Direc.'] = (df['dir'] * 360.0 / 100.0)

#  opcional: si quieres normalizar 360 -> 0 (valores en [0,360) en lugar de [0,360])
# df['Direc.'] = df['Direc.'] % 360

# 4) renombrar las otras columnas
rename_map = {
    "fecha": "datetime",
    "tmed": "Temp.",
    "velmedia": "Veloc."
}
df = df.rename(columns=rename_map)

# 5) eliminar la columna original 'dir' (ya tenemos 'Direc.')
df = df.drop(columns=['dir', 'indicativo', 'nombre', 'provincia', 'altitud', 'prec', ], errors='ignore')

# 6) asegurar formato de datetime (entrada en formato 'YYYY-MM-DD')
#    si hay filas con fecha inválida quedarán NaT -> luego se formatean como NaN en CSV
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce').dt.strftime("%Y-%m-%d")

# 7) reordenar columnas (opcional) para tener un CSV limpio
cols_desired = ['datetime', 'Temp.', 'Direc.', 'Veloc.']
# mantener cualquier otra columna adicional al final
other_cols = [c for c in df.columns if c not in cols_desired]
df = df[[c for c in cols_desired if c in df.columns]]

# 8) guardar el CSV resultante
df.to_csv(output_path, index=False)
print(f"Fichero guardado en: {output_path}")



Fichero guardado en: /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-AEMET/Datosmeteorologicossegorbe_limpios.csv


# COMBINA LAS VARIABLES MET (AEMET) CON LAS F-Q (GVA) EN UN UNICO CSV

In [2]:
import pandas as pd

# --- Ajusta estas rutas ---
csv_aires_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_media_diaria/df__Viver_diario_vfq.csv"     # CSV que contiene NO, NO2, NOx, O3
csv_meteo_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-AEMET/Datosmeteorologicossegorbe_limpios.csv"     # CSV que contiene Temp., Veloc., Direc.
output_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-VIVER/Datosmet+F-QViver_SIN_R.SOL.csv"     # CSV resultante
# ---------------------------

# Leer CSVs (sin parseo forzado aún)
df_aires = pd.read_csv(csv_aires_path)
df_meteo = pd.read_csv(csv_meteo_path)

# Función auxiliar para detectar y convertir la columna de fecha a 'datetime'
def ensure_datetime_col(df):
    # posibles nombres de la columna de fecha
    candidates = ["datetime", "fecha", "DateTime", "date", "Fecha"]
    found = None
    for c in candidates:
        if c in df.columns:
            found = c
            break
    if found is None:
        raise KeyError(f"No se encontró columna de fecha. Busqué: {candidates}")
    # convertir a datetime y renombrar a 'datetime'
    df = df.copy()
    df["datetime"] = pd.to_datetime(df[found], errors="coerce")
    # eliminar la columna original si se llamaba distinto
    if found != "datetime":
        df = df.drop(columns=[found])
    return df

# Aplicar conversión
df_aires = ensure_datetime_col(df_aires)
df_meteo = ensure_datetime_col(df_meteo)

# Comprobar columnas objetivo y construir lista de columnas a seleccionar
cols_aires = ["NO", "NO2", "NOx", "O3"]
cols_meteo = ["Temp.", "Veloc.", "Direc."]

avail_aires = [c for c in cols_aires if c in df_aires.columns]
avail_meteo = [c for c in cols_meteo if c in df_meteo.columns]

if not avail_aires:
    raise ValueError(f"No se encontraron columnas de contaminantes en {csv_aires_path}. Busqué: {cols_aires}")
if not avail_meteo:
    raise ValueError(f"No se encontraron columnas meteorológicas en {csv_meteo_path}. Busqué: {cols_meteo}")

# Seleccionar columnas necesarias (incluimos 'datetime' en ambos para merge)
df_aires_sel = df_aires[["datetime"] + avail_aires].copy()
df_meteo_sel = df_meteo[["datetime"] + avail_meteo].copy()

# Opcional: eliminar filas con datetime inválido (NaT)
df_aires_sel = df_aires_sel.dropna(subset=["datetime"])
df_meteo_sel = df_meteo_sel.dropna(subset=["datetime"])

# Asegurar que no haya duplicados de datetime (si los hay, los agrupamos tomando la media)
if df_aires_sel["datetime"].duplicated().any():
    df_aires_sel = df_aires_sel.groupby("datetime", as_index=False)[avail_aires].mean()
if df_meteo_sel["datetime"].duplicated().any():
    df_meteo_sel = df_meteo_sel.groupby("datetime", as_index=False)[avail_meteo].mean()

# Hacer el merge por 'datetime' (por defecto inner: solo comunes)
merged = pd.merge(df_aires_sel, df_meteo_sel, on="datetime", how="inner", validate="one_to_one")

# Si quieres un merge distinto (por ejemplo mantener todos los tiempos), cambia how="inner" por "outer" o "left"
# merged = pd.merge(df_aires_sel, df_meteo_sel, on="datetime", how="outer", validate="one_to_one")

# Opcional: ordenar por datetime
merged = merged.sort_values("datetime").reset_index(drop=True)

# Formatear datetime en el CSV de salida (ej: YYYY-MM-DD o con hora si la tienen)
# Si quieres que conserve hora: comentarlo para dejar datetime como datetime64 en el DataFrame pero al guardar se convierte a string.
merged["datetime"] = merged["datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Guardar CSV final
merged.to_csv(output_path, index=False)
print(f"CSV combinado guardado en: {output_path}")
print(f"Columnas incluidas: {list(merged.columns)}")
print(f"Nº filas resultantes: {len(merged)}")


CSV combinado guardado en: /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-VIVER/Datosmet+F-QViver_SIN_R.SOL.csv
Columnas incluidas: ['datetime', 'NO', 'NO2', 'NOx', 'O3', 'Temp.', 'Veloc.', 'Direc.']
Nº filas resultantes: 5077


# TRATAMIENTO VARIABLE R.SOL.

In [11]:
# CELDA ROBUSTA para notebook
# Lee CSV con header en la fila 9 (index 8) y, si hace falta, reconstruye la columna time
# Extrae la columna más plausible para Gb(i), parsea timestamps (incluye 2400), calcula media diaria y guarda CSV.
import os
from pathlib import Path
import pandas as pd
import re
from datetime import datetime, timedelta
import io

# ------------ CONFIGURA RUTAS ------------
input_path  = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos_R.Sol./Radiacion solar Viver.csv"   # <- pon tu ruta
output_path = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-VIVER/DatosViver_R.SOL.csv"
header_row = 8       # fila 9 -> index 8
# -----------------------------------------

# utilidades
def find_first_csv(base='.'):
    base = Path(base)
    csvs = list(base.rglob('*.csv'))
    return str(csvs[0]) if csvs else None

def read_raw_lines(path, n=200, encodings=('utf-8','latin1','cp1252')):
    for enc in encodings:
        try:
            with open(path, 'r', encoding=enc, errors='ignore') as f:
                lines = [next(f) for _ in range(n)]
            return lines, enc
        except StopIteration:
            # archivo corto: leer todo
            try:
                with open(path, 'r', encoding=enc, errors='ignore') as f:
                    lines = f.readlines()
                return lines, enc
            except Exception:
                continue
        except Exception:
            continue
    raise FileNotFoundError(f"No se pudo leer el archivo con los encodings probados: {encodings}")

def detect_sep_by_sample(lines):
    """
    Prueba separadores comunes en la línea de header (row index = header_row) y en las líneas de datos:
    Devuelve el sep con mayor puntuación según conteos de tokens que parecen fechas/hhmm.
    """
    candidates = [',',';','\t','|']
    scores = {c:0 for c in candidates}
    sample = lines[:200]
    date_re = re.compile(r'^\d{8}$')       # 20120101
    datetime_re = re.compile(r'^\d{8}[:]\d{4}$')  # 20120101:0130
    for c in candidates:
        for ln in sample:
            parts = ln.strip().split(c)
            if len(parts) >= 2:
                # primer token fecha? o primer token datetime?
                t0 = parts[0].strip()
                t1 = parts[1].strip() if len(parts)>1 else ''
                if date_re.match(t0) and re.match(r'^\d{3,4}$', re.sub(r'\D','',t1)):
                    scores[c] += 2
                if datetime_re.match(t0) or datetime_re.match(t0 + ':' + t1):
                    scores[c] += 3
                # si aparece un patrón tipo 'YYYYMMDD:HHMM' dentro del primer campo
                if re.search(r'\d{8}[:]\d{3,4}', ln):
                    scores[c] += 2
    # elegir el sep con mayor score (si todos 0, devuelve None)
    best = max(scores.items(), key=lambda kv: kv[1])
    if best[1] == 0:
        return None
    return best[0]

def try_pandas_read(path, sep, header_row, encoding='utf-8'):
    """Intento de lectura con pandas y on_bad_lines='skip'"""
    try:
        df = pd.read_csv(path, sep=sep, engine='python', header=header_row, encoding=encoding, on_bad_lines='skip', skip_blank_lines=True)
        return df
    except Exception as e:
        return None

# parser time robusto (manejo 2400)
def parse_time_string(s):
    if pd.isna(s):
        return pd.NaT
    s = str(s).strip()
    if not s:
        return pd.NaT
    # aceptar formatos como '20120101:0130', '20120101: 0130', '20120101', '0130', '20120101,0130' etc.
    # eliminar comillas
    s = s.replace('"','').replace("'",'')
    # buscar patrón completo YYYYMMDD:HHMM
    m = re.search(r'(\d{8})\s*[:,-]?\s*(\d{3,4})', s)
    if m:
        date_part = m.group(1)
        time_part = m.group(2).zfill(4)
    else:
        # si s es solo YYYYMMDD o solo HHMM, devolver NaT (se tratará combinando columnas)
        return pd.NaT
    try:
        year = int(date_part[0:4]); month = int(date_part[4:6]); day = int(date_part[6:8])
        hour = int(time_part[0:2]); minute = int(time_part[2:4])
    except Exception:
        return pd.NaT
    if hour == 24 and minute == 0:
        try:
            dt = datetime(year, month, day) + timedelta(days=1)
            return pd.Timestamp(dt.replace(hour=0, minute=0, second=0))
        except Exception:
            return pd.NaT
    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        return pd.NaT
    try:
        return pd.Timestamp(year=year, month=month, day=day, hour=hour, minute=minute, second=0)
    except Exception:
        return pd.NaT

# ---------- START ----------
if input_path is None:
    auto = find_first_csv('.')
    if auto is None:
        raise FileNotFoundError("No hay input_path y no se encontró ningún .csv en el árbol de trabajo.")
    input_path = auto
    print(f"No se proporcionó input_path. Usando primer CSV encontrado: {input_path}")

if not os.path.exists(input_path):
    raise FileNotFoundError(f"No existe el archivo: {input_path}")

# leer raw lines para análisis
lines, used_enc = read_raw_lines(input_path, n=300)
print(f"Leído sample de archivo con encoding sugerido: {used_enc}")
# opcional: imprime las primeras 12 líneas para inspección
print("\n--- Primeras 12 líneas del archivo ---")
for i, L in enumerate(lines[:12]):
    print(f"{i+1:02d}: {L.rstrip()}")
print("--- fin preview ---\n")

# detectar separador probable
sep = detect_sep_by_sample(lines)
if sep:
    print(f"Separador detectado por heurística: '{sep}'")
else:
    print("No se detectó separador claro. Intentaremos con [',',';','\\t','|'] mediante pandas fallback.")
    sep = None

# intentar lectura con header en la fila 9 (index 8)
df = None
encodings_to_try = [used_enc] if used_enc else []
encodings_to_try += ['utf-8','latin1','cp1252']
seps_to_try = [sep] if sep else []
seps_to_try += [',',';','\t','|']

for enc in encodings_to_try:
    for s in seps_to_try:
        if s is None: 
            continue
        df_try = try_pandas_read(input_path, s, header_row, encoding=enc)
        if df_try is not None and df_try.shape[1] >= 2:
            df = df_try
            chosen_sep = s
            chosen_enc = enc
            break
    if df is not None:
        break

if df is None:
    # fallback: leer sin header (header=None) con el mejor sep posible o con sep=','
    print("No se pudo leer con header en la fila 9. Se intenta lectura sin header (header=None) y reconstrucción manual.")
    # elegir sep por conteo simple
    if sep is None:
        sep = detect_sep_by_sample(lines) or ','
    try:
        df = pd.read_csv(input_path, sep=sep, engine='python', header=None, encoding=used_enc or 'latin1', on_bad_lines='skip', skip_blank_lines=True)
        chosen_sep = sep
        chosen_enc = used_enc or 'latin1'
        print(f"Leído con header=None, sep='{chosen_sep}', encoding='{chosen_enc}', df.shape={df.shape}")
    except Exception as e:
        raise ValueError(f"No se pudo leer el archivo aun con header=None. Error: {e}")

else:
    print(f"Lectura OK con sep='{chosen_sep}', encoding='{chosen_enc}'. df.shape = {df.shape}")
    # normalizar nombres
    df.columns = [str(c).strip().replace('"','').replace("'",'') for c in df.columns]

# Ahora: buscar columnas time y Gb(i)
def find_time_and_value_cols(df):
    cols = list(df.columns)
    # 1) si existen nombres y contienen 'time'/'gb'
    lower = [str(c).lower() for c in cols]
    time_col = None
    gb_col = None
    for i, c in enumerate(lower):
        if 'time' == c or 'time' in c:
            time_col = cols[i]
        if c == 'gb(i)' or 'gb(i)' in c or (c.startswith('gb') and 'gb' in c):
            gb_col = cols[i]
    if time_col and gb_col:
        return time_col, gb_col

    # 2) si df fue leído con header=None o no se detectaron nombres, intentar heurísticas por contenido
    # buscar columna con formato 'YYYYMMDD:HHMM' o 'YYYYMMDD' or HHMM
    time_candidates = []
    for col in cols:
        ser = df[col].astype(str).str.strip().fillna('')
        # contar filas con pattern date or datetime
        n_date = ser.str.match(r'^\d{8}$').sum()
        n_datetime = ser.str.match(r'^\d{8}[:,-]?\d{3,4}').sum()
        n_time = ser.str.match(r'^\d{3,4}$').sum()
        score = n_datetime*3 + n_date*2 + n_time
        if score > 0:
            time_candidates.append((col, score))
    time_candidates = sorted(time_candidates, key=lambda x: x[1], reverse=True)

    # buscar columna numérica plausible para Gb(i)
    num_scores = []
    for col in cols:
        ser = pd.to_numeric(df[col].astype(str).str.replace(',','.'), errors='coerce')
        frac_numeric = ser.notna().mean()
        num_scores.append((col, frac_numeric))
    num_scores = sorted(num_scores, key=lambda x: x[1], reverse=True)

    # heurística: si el mejor time_candidate existe, usarlo; si no, si las dos primeras columnas parecen fecha y hora separadas, combinarlas
    if time_candidates:
        time_col = time_candidates[0][0]
    else:
        # comprobar si col0 es YYYYMMDD y col1 es HHMM en muchas filas
        if df.shape[1] >= 2:
            col0 = df.columns[0]; col1 = df.columns[1]
            s0 = df[col0].astype(str).str.strip().fillna('')
            s1 = df[col1].astype(str).str.strip().fillna('')
            n0 = s0.str.match(r'^\d{8}$').sum()
            n1 = s1.str.match(r'^\d{3,4}$').sum()
            if n0 > 0.6 * len(s0) and n1 > 0.6 * len(s1):
                # asumimos combinación col0:col1
                time_col = (col0, col1)  # retornamos tupla para indicar combinación
    # elegir gb_col como columna con mayor frac_numeric, evitando la(s) columna(s) usadas para time
    gb_col = None
    for col, frac in num_scores:
        # si time_col es tuple y coincide con col, saltar
        if isinstance(time_col, tuple) and col in time_col:
            continue
        # si time_col string and equals, saltar
        if isinstance(time_col, str) and col == time_col:
            continue
        # preferir columnas cuyo nombre contenga 'gb' si hay empate o buena fracción
        if 'gb' in str(col).lower():
            gb_col = col
            break
        if gb_col is None:
            gb_col = col
    return time_col, gb_col

time_col, gb_col = find_time_and_value_cols(df)

# si time_col es tupla -> hay que combinar
combined_time_from_two = False
if isinstance(time_col, tuple):
    col_date, col_time = time_col
    print(f"Detectada fecha en col '{col_date}' y hora en col '{col_time}', se combinarán como 'YYYYMMDD:HHMM'.")
    # crear columna temporal combinada
    def combine_row(d, t):
        da = str(d).strip()
        ti = str(t).strip()
        if not da or not ti:
            return ''
        # extraer primeros 4 dígitos de la parte de hora si viene pegada con comas
        m = re.match(r'(\d{3,4})', ti)
        if m:
            ti_clean = m.group(1).zfill(4)
        else:
            ti_clean = ti.zfill(4)
        return da + ':' + ti_clean
    df['_time_combined_raw'] = df[col_date].astype(str).str.strip() + ':' + df[col_time].astype(str).str.strip()
    # mejor: aplicar combine_row fila a fila
    df['_time_combined_raw'] = df.apply(lambda row: combine_row(row[col_date], row[col_time]), axis=1)
    df['_time_parsed'] = df['_time_combined_raw'].apply(parse_time_string)
    combined_time_from_two = True
else:
    # time_col es nombre de columna o None
    if time_col is None:
        print("No se detectó automáticamente una columna de tiempo. Voy a intentar extraer tiempo desde el primer token de la primera columna.")
        # intentar extraer con regex desde el inicio de la primera columna
        first_col = df.columns[0]
        df['_time_combined_raw'] = df[first_col].astype(str).str.extract(r'(^\d{8}[:,-]?\d{3,4})', expand=False).fillna('')
        df['_time_parsed'] = df['_time_combined_raw'].apply(parse_time_string)
    else:
        print(f"Usando columna detectada para tiempo: '{time_col}'")
        df['_time_parsed'] = df[time_col].apply(parse_time_string)

# si no hay suficientes parsed, intentar extraer datetime desde el comienzo de la fila completa
n_parsed = df['_time_parsed'].notna().sum()
if n_parsed < 1:
    print("Pocos/ningún timestamp parseable. Intentaré extraer patrón 'YYYYMMDD:HHMM' del comienzo de cada línea raw.")
    raw_lines = [l.rstrip('\n') for l in lines]
    # crear serie con regex aplicada a raw lines
    extracted = []
    for i, row in df.iterrows():
        try:
            rawline = raw_lines[i] if i < len(raw_lines) else ','.join(map(str, row.values.tolist()))
        except Exception:
            rawline = ','.join(map(str, row.values.tolist()))
        m = re.search(r'(\d{8}[:,-]?\d{3,4})', rawline)
        extracted.append(m.group(1) if m else '')
    df['_time_combined_raw2'] = extracted
    df['_time_parsed'] = df['_time_combined_raw2'].apply(parse_time_string)
    n_parsed = df['_time_parsed'].notna().sum()

print(f"Timestamps parseados con éxito: {n_parsed} filas (de {len(df)})")

# detectar columna Gb(i) (si gb_col es None, elegir columna numérica con mayor fracción de no-NaN)
if gb_col is None:
    print("No se detectó columna con nombre 'Gb(i)'. Seleccionando la columna más numérica disponible como Gb(i).")
    numeric_fracs = []
    for col in df.columns:
        if col.startswith('_'): 
            continue
        ser = pd.to_numeric(df[col].astype(str).str.replace(',','.'), errors='coerce')
        numeric_fracs.append((col, ser.notna().mean()))
    numeric_fracs = sorted(numeric_fracs, key=lambda x: x[1], reverse=True)
    if numeric_fracs:
        gb_col = numeric_fracs[0][0]
        print(f"Se seleccionó '{gb_col}' como columna Gb(i) (fracción numérica = {numeric_fracs[0][1]:.2f})")
    else:
        raise KeyError("No se encontró ninguna columna numérica para usar como Gb(i).")

# convertir Gb(i) a numérico
df[gb_col] = pd.to_numeric(df[gb_col].astype(str).str.replace(',','.'), errors='coerce')

# agrupar por día y calcular media ignorando NaN
df['_date'] = df['_time_parsed'].dt.date
daily = df.groupby('_date', dropna=True)[gb_col].mean().reset_index()
daily = daily.rename(columns={gb_col: 'R.Sol.'})
daily['datetime'] = daily['_date'].astype(str)
output_df = daily[['datetime','R.Sol.']]

# guardar resultado
output_df.to_csv(output_path, index=False)
print(f"\nCSV diario generado con {len(output_df)} filas -> {output_path}")
print("Columnas usadas: time(parsed) -> _time_parsed ; Gb(i) ->", gb_col)
if combined_time_from_two:
    print("Aviso: la columna time fue reconstruida combinando las dos primeras columnas (fecha + hora).")


Leído sample de archivo con encoding sugerido: utf-8

--- Primeras 12 líneas del archivo ---
01: Latitude (decimal degrees):	39.922
02: Longitude (decimal degrees):	-0.599
03: Elevation (m):	551
04: Radiation database:	PVGIS-ERA5
05: 
06: 
07: Slope: 0 deg.
08: Azimuth: 0 deg.
09: time,Gb(i),Gd(i),Gr(i),H_sun,T2m,WS10m,Int
10: 20120101:0030,0.0,0.0,0.0,0.0,8.49,3.24,0.0
11: 20120101:0130,0.0,0.0,0.0,0.0,8.09,3.17,0.0
12: 20120101:0230,0.0,0.0,0.0,0.0,7.72,3.1,0.0
--- fin preview ---

Separador detectado por heurística: ','
Lectura OK con sep=',', encoding='utf-8'. df.shape = (105198, 8)
Usando columna detectada para tiempo: '20120101:0130'
Timestamps parseados con éxito: 105190 filas (de 105198)

CSV diario generado con 4383 filas -> /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-VIVER/DatosViver_R.SOL.csv
Columnas usadas: time(parsed) -> _time_parsed ; Gb(i) -> 0.0


# JUNTAR DATOS + R.SOL.

In [3]:
# Requiere: pandas
import pandas as pd
from pathlib import Path

# ----------------- CONFIGURA RUTAS -----------------
air_csv_path   = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-PALANCIA/Datosmet+F-Qpalancia_SIN_R.SOL.csv"    # CSV con columnas ['datetime','NO','NO2','NOx','O3','Temp.','Veloc.','Direc.']
rsol_csv_path  = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datos-PALANCIA/Datospalancia_R.SOL.csv"    # CSV con columnas ['datetime','R.Sol.']
output_path    = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/DIARIOS/Palancia.csv"
# join_type: 'inner' -> solo filas con datetime en ambos CSV
#            'left'  -> todas las filas del CSV de aire, R.Sol. quedará NaN si no hay coincidencia
join_type = 'left'
# --------------------------------------------------

def read_csv_flexible(path: str) -> pd.DataFrame:
    """Leer CSV intentando que pandas detecte el separador correctamente."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")
    # intentamos inferir separador con sep=None (engine python)
    try:
        df = pd.read_csv(path, sep=None, engine='python', on_bad_lines='skip')
        return df
    except Exception:
        # fallback: probar separadores comunes
        for sep in [',',';','\t','|']:
            try:
                df = pd.read_csv(path, sep=sep, engine='python', on_bad_lines='skip')
                return df
            except Exception:
                continue
        # si todo falla, lanzar
        raise

def parse_datetime_column(df: pd.DataFrame, col_name: str = 'datetime') -> pd.DataFrame:
    """Intentar parsear de forma robusta la columna datetime y devolver df con la columna en dtype datetime64[ns]."""
    if col_name not in df.columns:
        raise KeyError(f"No existe la columna '{col_name}' en el DataFrame.")
    s = df[col_name].astype(str).str.strip()
    # intento 1: parse directo con inferencia
    parsed = pd.to_datetime(s, infer_datetime_format=True, dayfirst=False, errors='coerce')
    # si muchos NaT, intentar dayfirst=True
    n_total = len(parsed)
    n_nat = parsed.isna().sum()
    if n_total > 0 and n_nat / n_total > 0.05:  # si >5% no parseadas, intentar con dayfirst
        parsed2 = pd.to_datetime(s, infer_datetime_format=True, dayfirst=True, errors='coerce')
        if parsed2.notna().sum() > parsed.notna().sum():
            parsed = parsed2
            n_nat = parsed.isna().sum()
    # intentar formatos comunes adicionales si sigue habiendo NaT
    if n_total > 0 and n_nat / n_total > 0.05:
        common_formats = [
            '%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M', '%Y/%m/%d %H:%M:%S', '%Y/%m/%d %H:%M',
            '%d/%m/%Y %H:%M:%S', '%d/%m/%Y %H:%M', '%Y%m%d:%H%M', '%Y%m%d%H%M', '%Y%m%d %H%M',
            '%Y-%m-%d', '%d/%m/%Y'
        ]
        for fmt in common_formats:
            parsed_try = pd.to_datetime(s, format=fmt, errors='coerce')
            # si mejora el número de parseados, adoptarlo
            if parsed_try.notna().sum() > parsed.notna().sum():
                parsed = parsed_try
                n_nat = parsed.isna().sum()
            if n_nat / n_total <= 0.05:
                break
    # asignar columna parseada
    df = df.copy()
    df[col_name] = parsed
    # informar si quedan no-parseables
    n_nat_final = df[col_name].isna().sum()
    if n_nat_final > 0:
        print(f"Aviso: {n_nat_final} filas en '{col_name}' no se pudieron parsear y quedarán como NaT.")
    return df

# ---------- Lectura ----------
df_air = read_csv_flexible(air_csv_path)
df_rsol = read_csv_flexible(rsol_csv_path)

# Normalizar nombres (quitar espacios alrededor)
df_air.columns = [c.strip() for c in df_air.columns]
df_rsol.columns = [c.strip() for c in df_rsol.columns]

# Comprobar columnas esperadas y avisar
expected_air_cols = ['datetime','NO','NO2','NOx','O3','Temp.','Veloc.','Direc.']
missing_air = [c for c in expected_air_cols if c not in df_air.columns]
if missing_air:
    print(f"Aviso: faltan columnas esperadas en el CSV de aire: {missing_air}. El script seguirá intentando con las columnas disponibles.")

if 'datetime' not in df_rsol.columns or 'R.Sol.' not in df_rsol.columns:
    print("Aviso: en el CSV de R.Sol. se esperaba ['datetime','R.Sol.']. Revisa los encabezados.")

# ---------- Parsear datetime ----------
df_air = parse_datetime_column(df_air, 'datetime')
df_rsol = parse_datetime_column(df_rsol, 'datetime')

# Opcional: si quieres forzar que las horas/minutos sean exactamente a la hora (p. ej. truncar minutos),
# descomentarlo. Por defecto no se altera.
# df_air['datetime'] = df_air['datetime'].dt.floor('min')
# df_rsol['datetime'] = df_rsol['datetime'].dt.floor('min')

# ---------- Unir ----------
# ordenar por datetime por claridad
df_air = df_air.sort_values('datetime').reset_index(drop=True)
df_rsol = df_rsol.sort_values('datetime').reset_index(drop=True)

# hacemos merge en la columna 'datetime'
merged = pd.merge(df_air, df_rsol[['datetime','R.Sol.']], on='datetime', how=join_type)

# Reordenar columnas en el orden solicitado, si existen
final_cols = ['datetime','NO','NO2','NOx','O3','Temp.','Veloc.','Direc.','R.Sol.']
# mantener solo las que existan en merged y en ese orden
final_cols_existing = [c for c in final_cols if c in merged.columns]
merged = merged[final_cols_existing]

# ---------- Guardar ----------
# por defecto separador ','. Cambia sep=';' si lo prefieres.
merged.to_csv(output_path, index=False)

print(f"Merge completado. Resultado guardado en: {output_path}")
print(f"Filas air: {len(df_air)}, filas rsol: {len(df_rsol)}, filas resultado ({join_type}): {len(merged)}")


Merge completado. Resultado guardado en: /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/DIARIOS/Palancia.csv
Filas air: 5006, filas rsol: 4383, filas resultado (left): 5006


/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_73935/2317535412.py:40: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(s, infer_datetime_format=True, dayfirst=False, errors='coerce')
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_73935/2317535412.py:40: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(s, infer_datetime_format=True, dayfirst=False, errors='coerce')
